# CROPSPECTRA — Phase B: Rice Leaf Disease Image Classifier

**Project:** CROPSPECTRA — Autonomous Crop Inspection Rover  
**Phase:** B — Computer Vision Model Training (MobileNetV3 Transfer Learning)  
**Dataset:** Rice Leaf Diseases (`archive1`: 2,628 images across 6 balanced classes)  

---
### Overview
This notebook trains an edge-optimized Deep Learning classifier using **MobileNetV3** for real-time rice leaf disease detection onboard the autonomous inspection rover.

**Target Classes (6):**
1. `bacterial_leaf_blight`
2. `brown_spot`
3. `healthy`
4. `leaf_blast`
5. `leaf_scald`
6. `narrow_brown_spot`

## Step 1: Import Libraries and Verify Environment

In [ ]:
import os
import sys
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device configured: {device}')

## Step 2: Explore Dataset and Visualize Samples

In [ ]:
TRAIN_DIR = '../DATASET/archive1/RiceLeafsDisease/train'
VAL_DIR = '../DATASET/archive1/RiceLeafsDisease/validation'

classes = sorted(os.listdir(TRAIN_DIR))
print(f'Found {len(classes)} classes:')
for c in classes:
    count = len(os.listdir(os.path.join(TRAIN_DIR, c)))
    print(f'  • {c:25s}: {count} train images')

# Display one sample from each class
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, cls in enumerate(classes):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    sample_img = os.listdir(cls_dir)[0]
    img = Image.open(os.path.join(cls_dir, sample_img))
    axes[i].imshow(img)
    axes[i].set_title(cls.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## Step 3: Run Training Script
We use the modular training script [`src/train_image_classifier.py`](../src/train_image_classifier.py) which handles data augmentation, cosine learning rate scheduling, model checkpointing, and evaluation metrics.

In [ ]:
!python ../src/train_image_classifier.py --epochs 8 --batch-size 32 --arch mobilenet_v3_small

## Step 4: Display Training Curves & Confusion Matrix

In [ ]:
metrics_img = '../reports/training_metrics.png'
cm_img = '../reports/confusion_matrix.png'

if os.path.exists(metrics_img):
    plt.figure(figsize=(12, 5))
    plt.imshow(Image.open(metrics_img))
    plt.axis('off')
    plt.title('Training & Validation Curves', fontsize=14)
    plt.show()

if os.path.exists(cm_img):
    plt.figure(figsize=(8, 7))
    plt.imshow(Image.open(cm_img))
    plt.axis('off')
    plt.title('Validation Confusion Matrix', fontsize=14)
    plt.show()

## Step 5: Test Model Inference & Autonomous Rover Disease Advisory Engine

In [ ]:
from pathlib import Path
# Test prediction on a sample validation image
sample_test = str(list(Path('../DATASET/archive1/RiceLeafsDisease/validation/leaf_blast').glob('*.JPG'))[0])
print(f'Testing on: {sample_test}')
!python ../src/predict.py --image "{sample_test}"